# Data Preparation — NautiCost

This notebook cleans and structures the raw NautiCost data files and exports ready-to-use CSVs.

**Input files (raw):**
- `Kostnader_MM.csv` — transaction-level cost/charge data (all yachts, 2020–2025)
- `Yacht-specs.csv` — physical and operational specifications per yacht
- `cockpit_2020.csv` … `cockpit_2025.csv` — aggregated KPIs per year

**Output files (clean):**
- `costs_clean.csv` — parsed, typed, flagged transaction data
- `specs_clean.csv` — validated and typed yacht specifications
- `cockpit_clean.csv` — one row per year with revenue, gross margin, and network KPIs

In [ ]:
import pandas as pd
import numpy as np
import re
import csv
from pathlib import Path

DATA_DIR = Path('../004 data')
print('Data directory:', DATA_DIR.resolve())

---
## 1. Kostnader_MM.csv — Transaction Data

### 1.1 Known Raw Data Issues

| Issue | Location | Fix |
|-------|----------|-----|
| `Yacht:_2` (colon typo) | ~line 2637 | Replace with `Yacht_2` |
| `Yacht:_14` (colon typo) | ~line 3222 | Replace with `Yacht_14` |
| Subtotal rows (no dates, only amounts) | After each transaction group | Skip — do not parse as transactions |
| `Not set` in VAT columns | Systematic | Treat as NaN — VAT not tracked |
| Two column layouts (standard vs extended with `viewedit` prefix) | Sections vary | Detect by col[8] content |

In [ ]:
def looks_like_money(s: str) -> bool:
    """True if string represents a monetary value (e.g. '3,665.83' or '0.00')."""
    s = s.strip()
    return bool(re.match(r'^[\d,]+\.\d{2}$', s))

def parse_money(s) -> float:
    """Parse '3,665.83' or '0.00' to float; returns NaN for missing/non-numeric."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s in ('', 'Not set'):
        return np.nan
    try:
        return float(s.replace(',', ''))
    except ValueError:
        return np.nan

date_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')
yacht_id_pattern = re.compile(r'^Yacht[_:]\d+$', re.IGNORECASE)

records = []
current_yacht = None
issues_found = []

with open(DATA_DIR / 'Kostnader_MM.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for lineno, row in enumerate(reader, start=1):
        if not row or len(row) < 4:
            continue

        first = row[0].strip()

        # --- Detect and fix yacht ID header rows ---
        if yacht_id_pattern.match(first):
            raw_id = first
            # Fix colon typo: Yacht:_2 → Yacht_2
            fixed_id = re.sub(r'Yacht[_:](\d+)', lambda m: f'yacht_{m.group(1)}', raw_id, flags=re.IGNORECASE)
            if raw_id != fixed_id.upper().replace('YACHT_', 'Yacht_'):
                issues_found.append(f'Line {lineno}: Fixed yacht ID "{raw_id}" → "{fixed_id}"')
            current_yacht = fixed_id.lower()
            continue

        # Skip column header rows
        if first in ('#', 'Clear') or 'Service Type' in str(row[1]):
            continue
        if current_yacht is None:
            continue

        # Must have a date in col 3 to be a transaction row
        if not (len(row) > 3 and date_pattern.match(str(row[3]).strip())):
            continue

        # Detect format: standard (col[0] empty) vs extended (col[0] = 'viewedit')
        col8 = row[8].strip() if len(row) > 8 else ''
        is_standard = looks_like_money(col8) or col8 in ('0.00', '')

        try:
            if is_standard:
                office          = row[1].strip()
                arrival_port    = row[2].strip()
                arrival_date    = row[3].strip()
                departure_date  = row[4].strip()
                service_type    = row[5].strip()
                invoice_comments= row[6].strip()
                supplier        = row[7].strip()
                cost_no_vat     = row[8].strip()
                final_charge    = row[15].strip() if len(row) > 15 else ''
            else:
                # Extended format: col[0] = 'viewedit', one extra column shifts everything
                office          = row[1].strip()
                arrival_port    = row[2].strip()
                arrival_date    = row[3].strip()
                departure_date  = row[4].strip()
                service_type    = row[5].strip()
                invoice_comments= row[7].strip()
                supplier        = row[8].strip()
                cost_no_vat     = row[9].strip()
                final_charge    = row[16].strip() if len(row) > 16 else ''

            records.append({
                'yacht_id'        : current_yacht,
                'office'          : office,
                'arrival_port'    : arrival_port,
                'arrival_date'    : arrival_date,
                'departure_date'  : departure_date,
                'service_type'    : service_type,
                'invoice_comments': invoice_comments,
                'supplier'        : supplier,
                'cost_no_vat'     : cost_no_vat,
                'final_charge'    : final_charge,
                'row_format'      : 'standard' if is_standard else 'extended',
            })
        except IndexError:
            issues_found.append(f'Line {lineno}: Too few columns ({len(row)}), skipped')

costs_raw = pd.DataFrame(records)
print(f'Parsed rows: {len(costs_raw)}')
print(f'\nIssues fixed during parsing:')
for issue in issues_found:
    print(' ', issue)

### 1.2 Type Conversion and Derived Columns

In [ ]:
costs = costs_raw.copy()

# Numeric columns
costs['cost_no_vat']  = costs['cost_no_vat'].apply(parse_money)
costs['final_charge'] = costs['final_charge'].apply(parse_money)

# Dates
costs['arrival_date']   = pd.to_datetime(costs['arrival_date'],   errors='coerce')
costs['departure_date'] = pd.to_datetime(costs['departure_date'], errors='coerce')

# Derived columns
costs['year']      = costs['arrival_date'].dt.year
costs['month']     = costs['arrival_date'].dt.month
costs['stay_days'] = (costs['departure_date'] - costs['arrival_date']).dt.days
costs['margin']    = costs['final_charge'] - costs['cost_no_vat']

# Categorical columns
costs['office']       = costs['office'].astype('category')
costs['service_type'] = costs['service_type'].astype('category')

print('Datatypes:')
print(costs.dtypes)
print(f'\nYear distribution:')
print(costs['year'].value_counts().sort_index())

### 1.3 Validation

In [ ]:
print('=== MISSING VALUES ===')
print(costs.isnull().sum())

print('\n=== NEGATIVE STAY DAYS ===')
neg_stay = costs[costs['stay_days'] < 0]
print(f'{len(neg_stay)} rows with negative stay (departure before arrival)')
if len(neg_stay) > 0:
    print(neg_stay[['yacht_id', 'arrival_date', 'departure_date', 'service_type']].head(10))

print('\n=== NEGATIVE/ZERO FINAL CHARGE ===')
zero_charge = costs[costs['final_charge'] <= 0]
print(f'{len(zero_charge)} rows with final_charge <= 0')
if len(zero_charge) > 0:
    print(zero_charge[['yacht_id', 'year', 'service_type', 'cost_no_vat', 'final_charge']].head(10))

print('\n=== UNKNOWN YACHT IDs ===')
known_yachts = {f'yacht_{i}' for i in range(1, 18)}
unknown = costs[~costs['yacht_id'].isin(known_yachts)]['yacht_id'].unique()
print(f'Unknown yacht IDs: {unknown}')

print('\n=== ROW FORMAT DISTRIBUTION ===')
print(costs['row_format'].value_counts())

In [ ]:
# Flag rows with data quality concerns (do not remove — flag for transparency)
costs['flag'] = ''
costs.loc[costs['final_charge'].isna(), 'flag'] += 'missing_charge;'
costs.loc[costs['cost_no_vat'].isna(), 'flag'] += 'missing_cost;'
costs.loc[costs['arrival_date'].isna(), 'flag'] += 'missing_date;'
costs.loc[(costs['stay_days'].notna()) & (costs['stay_days'] < 0), 'flag'] += 'negative_stay;'
costs.loc[(costs['final_charge'].notna()) & (costs['final_charge'] == 0), 'flag'] += 'zero_charge;'

n_flagged = (costs['flag'] != '').sum()
print(f'Flagged rows: {n_flagged} / {len(costs)}')
print(costs[costs['flag'] != ''][['yacht_id', 'year', 'service_type', 'final_charge', 'flag']].head(20))

---
## 2. Yacht-specs.csv — Specifications

### 2.1 Known Issues

| Yacht | Issue | Action |
|-------|-------|--------|
| yacht_17 | GT = 0 | Flag as suspect; set to NaN |
| yacht_1 | Air Draft = 0 | Flag as suspect; set to NaN |
| yacht_7 | NT missing | Keep as NaN |
| yacht_9, yacht_13, yacht_16, yacht_17 | Air Draft missing | Keep as NaN |
| yacht_2-1/2-2/2-3 | Three variants for one cost ID | Keep variants; create `yacht_id_base = yacht_2` for joining |

In [ ]:
specs = pd.read_csv(DATA_DIR / 'Yacht-specs.csv')
specs = specs.dropna(how='all').reset_index(drop=True)
specs = specs.loc[:, ~specs.columns.str.startswith('Unnamed')].copy()

# Normalize yacht ID
specs['yacht_id']      = specs['Yacht'].str.strip().str.lower()
specs['yacht_id_base'] = specs['yacht_id'].str.replace(r'-\d+$', '', regex=True)

# Numeric columns
num_cols = ['GT', 'NT', 'LOA (m)', 'Reg. Length (m)', 'Beam (m)', 'Draft (m)',
            'Air Draft (m)', 'Fuel consumption (L/h)']
for col in num_cols:
    specs[col] = pd.to_numeric(specs[col], errors='coerce')

print('Loaded specs:')
specs[['yacht_id', 'GT', 'NT', 'LOA (m)', 'Air Draft (m)', 'Fuel consumption (L/h)',
       'Loskrav (LOA>70m)', 'Størrelses-kategori']].to_string()
specs[['yacht_id', 'GT', 'NT', 'LOA (m)', 'Air Draft (m)', 'Fuel consumption (L/h)',
       'Loskrav (LOA>70m)', 'Størrelses-kategori']]

In [ ]:
specs_clean = specs.copy()

# Flag suspect zero values before replacing
suspect_zeros = []

# GT = 0 on yacht_17 is almost certainly a data entry error
mask_gt0 = (specs_clean['GT'] == 0)
if mask_gt0.any():
    suspect_zeros.append(specs_clean.loc[mask_gt0, 'yacht_id'].tolist())
    print(f'GT = 0 (set to NaN): {specs_clean.loc[mask_gt0, "yacht_id"].tolist()}')
    specs_clean.loc[mask_gt0, 'GT'] = np.nan

# Air Draft = 0 on yacht_1 — no superyacht has zero air draft; likely missing
mask_ad0 = (specs_clean['Air Draft (m)'] == 0)
if mask_ad0.any():
    print(f'Air Draft = 0 (set to NaN): {specs_clean.loc[mask_ad0, "yacht_id"].tolist()}')
    specs_clean.loc[mask_ad0, 'Air Draft (m)'] = np.nan

# Categorical — standardize casing
specs_clean['Størrelses-kategori'] = specs_clean['Størrelses-kategori'].str.strip().str.capitalize()
specs_clean['Loskrav (LOA>70m)']   = specs_clean['Loskrav (LOA>70m)'].str.strip().str.capitalize()

print('\nMissing values after cleaning:')
print(specs_clean[num_cols + ['Loskrav (LOA>70m)', 'Størrelses-kategori']].isnull().sum())

In [ ]:
# Sanity checks
print('=== GT vs LOA CONSISTENCY ===')
# GT should generally increase with LOA
print(specs_clean[['yacht_id', 'GT', 'LOA (m)', 'Størrelses-kategori']].sort_values('LOA (m)').to_string(index=False))

print('\n=== LOSKRAV CHECK (should be Ja for LOA > 70m) ===')
loskrav_check = specs_clean[['yacht_id', 'LOA (m)', 'Loskrav (LOA>70m)']].copy()
loskrav_check['expected_loskrav'] = loskrav_check['LOA (m)'].apply(
    lambda x: 'Ja' if pd.notna(x) and x > 70 else 'Nei'
)
mismatch = loskrav_check[loskrav_check['Loskrav (LOA>70m)'] != loskrav_check['expected_loskrav']]
if len(mismatch) > 0:
    print('Mismatches found:')
    print(mismatch)
else:
    print('All loskrav values consistent with LOA.')

print('\n=== FUEL CONSUMPTION COVERAGE ===')
no_fuel = specs_clean[specs_clean['Fuel consumption (L/h)'].isna()]['yacht_id'].tolist()
print(f'Yachts missing fuel consumption: {no_fuel}')

---
## 4. Export Cleaned Files

---
## 3. Cockpit CSVs (2020–2025) — Aggregated KPIs

### 3.1 Known Issues

| Issue | Detail | Action |
|-------|--------|--------|
| Avg. days per call/boat | Stored as Excel serial dates in CSV → unreadable | Skip these columns |
| 2020 format shift | Labels in col[0] instead of col[1] | Detected automatically |
| Empty/zero values for early years | 2020 had minimal activity | Kept as NaN / 0 |

In [ ]:
SERVICE_TYPES = {'Agency Fee', 'Agency Services', 'Bunkering', 'Hospitality',
                 'Port Marina', 'Provisioning', 'Technical Services', 'Total'}

SECTION_HEADERS = {
    'Revenues':        'revenue',
    'Gross Margin':    'gm',
    'Gross Margin (%)':'gm_pct',
}

def _to_float(s: str):
    """Convert string to float; return NaN for empty/non-numeric/date-like values."""
    s = str(s).strip()
    if not s or s in ('N/A', 'Not set'):
        return np.nan
    # Skip values that look like dates (Excel serial date artefacts)
    if re.match(r'\d{4}-\d{2}-\d{2}', s):
        return np.nan
    try:
        return float(s.replace(',', ''))
    except ValueError:
        return np.nan

def parse_cockpit_file(filepath: Path, year: int) -> dict:
    """
    Parse one cockpit CSV into a flat dict of KPIs.

    Column layout varies by year:
    - 2020: label in col[0], full-year actual in col[1]
    - 2021+: empty col[0], label in col[1], full-year actual in col[2]
    We auto-detect by finding the first non-empty cell per row.
    """
    raw = pd.read_csv(filepath, header=None, dtype=str).fillna('')
    section = None
    result = {'year': year}

    for _, row in raw.iterrows():
        vals = [v.strip() for v in row]

        # Find first non-empty cell (label) and next non-empty cell (value)
        label, value = '', ''
        for i, v in enumerate(vals):
            if v:
                label = v
                # Full-year actual is in the next non-empty cell
                for j in range(i + 1, len(vals)):
                    if vals[j]:
                        value = vals[j]
                        break
                break

        if not label:
            continue

        # Update section state
        if label in SECTION_HEADERS:
            section = SECTION_HEADERS[label]
            continue

        # Reset section on unrelated headers
        if label in ('Network Size and Activity', 'Size of Network',
                     'Value of Services', 'Network Activity',
                     'Avg. Gross Margin per Boat', 'Avg. Gross Margin per Call'):
            section = 'network'
            continue

        # Service type rows — store under current section
        if section and label in SERVICE_TYPES:
            col_name = f'{section}_{label.replace(" ", "_").replace("/", "_")}'
            result[col_name] = _to_float(value)
            continue

        # Network KPIs
        if 'unique boats' in label.lower():
            result['unique_boats'] = _to_float(value)
        elif 'total number of port calls' in label.lower():
            result['port_calls'] = _to_float(value)
        elif 'days spent in network' in label.lower():
            result['days_in_network'] = _to_float(value)
        # Skip avg days per call/boat — unreliable Excel date artefacts

    return result

# Parse all years
cockpit_records = []
for year in range(2020, 2026):
    fp = DATA_DIR / f'cockpit_{year}.csv'
    if fp.exists():
        rec = parse_cockpit_file(fp, year)
        cockpit_records.append(rec)
        print(f'{year}: {len(rec)-1} KPIs extracted')
    else:
        print(f'{year}: file not found — skipped')

cockpit = pd.DataFrame(cockpit_records).set_index('year').sort_index()
print(f'\nShape: {cockpit.shape}')
cockpit

### 3.2 Validation

In [ ]:
print('=== MISSING VALUES ===')
print(cockpit.isnull().sum())

print('\n=== REVENUE TOTALS PER YEAR ===')
if 'revenue_Total' in cockpit.columns:
    print(cockpit['revenue_Total'])

print('\n=== NETWORK ACTIVITY ===')
net_cols = [c for c in ('unique_boats', 'port_calls', 'days_in_network') if c in cockpit.columns]
print(cockpit[net_cols])

In [ ]:
# Export cleaned costs (all years)
costs_out = costs.drop(columns=['row_format'])
costs_out.to_csv(DATA_DIR / 'costs_clean.csv', index=False, encoding='utf-8')
print(f'Exported costs_clean.csv     — {len(costs_out)} rows')

# Export cleaned specs
specs_out = specs_clean.drop(columns=['yacht_id', 'yacht_id_base'])
specs_out.to_csv(DATA_DIR / 'specs_clean.csv', index=False, encoding='utf-8')
print(f'Exported specs_clean.csv     — {len(specs_out)} rows')

# Export cockpit KPIs
cockpit.to_csv(DATA_DIR / 'cockpit_clean.csv', encoding='utf-8')
print(f'Exported cockpit_clean.csv   — {len(cockpit)} rows (years), {len(cockpit.columns)} KPI columns')

---
## 5. Data Quality Summary

| File | Issue | Count | Action |
|------|-------|-------|--------|
| Kostnader_MM.csv | Colon typo i yacht ID (`Yacht:_2`, `Yacht:_14`) | 2 | Fikset under parsing |
| Kostnader_MM.csv | Subtotal-rader (ingen dato, bare beløp) | Mange | Hoppet over |
| Kostnader_MM.csv | `Not set` i VAT-kolonner | ~1 656 | Konvertert til NaN |
| Kostnader_MM.csv | Manglende/nullverdi i `final_charge` eller `cost_no_vat` | Se `flag`-kolonne | Flagget |
| Yacht-specs.csv | GT = 0 (yacht_17) | 1 | Satt til NaN |
| Yacht-specs.csv | Air Draft = 0 (yacht_1) | 1 | Satt til NaN |
| Yacht-specs.csv | NT mangler | yacht_7, yacht_17 | Beholdt som NaN |
| Yacht-specs.csv | Air Draft mangler | yacht_9, yacht_13, yacht_16, yacht_17 | Beholdt som NaN |
| Yacht-specs.csv | Fuel consumption mangler | Se validering | Beholdt som NaN |
| cockpit_YYYY.csv | Avg. days per call/boat som Excel-seriedatoer | Alle år | Kolonnene hoppet over |
| cockpit_YYYY.csv | 2020-format har label i col[0] (ikke col[1]) | 2020 | Håndtert automatisk |